In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc 
import anndata as ad
#import besca as bc
import h5py 
import glob
import matplotlib.pyplot as plt
import os

In [ ]:
sc.settings.verbosity = 3             # verbosity: errors (0), warnings (1), info (2), hints (3)
sc.logging.print_header()
sc.settings.set_figure_params(dpi=80, facecolor='white')
sc.set_figure_params(scanpy=True, figsize=(4,4))      #set scanpy plotting size

In [ ]:
base_path = '/home/EOCRC_atlas/'

# Annotate B Cells 

In [ ]:
# Load raw data for all cells
adata_raw = sc.read_h5ad(os.path.join(base_path, 'data/all_samples_raw_withAnnotation.h5ad'))

In [ ]:
# Cell type variable 
cell_type = 'B'

In [ ]:
# Subcluster and annotate the stromal cells
adata = adata_raw.copy()
del adata_raw 

adata = adata[adata.obs['Annotation_Tier1']==cell_type]
print(adata.shape)

In [ ]:
count_sum = adata.X.sum(axis=1)
# Normalize the data and store normalized data as its own layer for each normalization step 
adata.layers['counts'] = adata.X.copy()
sc.pp.normalize_total(adata, inplace = True, target_sum=1e4)
adata.layers['norm_counts'] = adata.X.copy()
sc.pp.log1p(adata)
adata.layers['log_counts'] = adata.X.copy()

In [ ]:
# check that counts layer is actually integers 
print(adata.layers['counts'][0:20,0:20])
print(adata.layers['norm_counts'][0:20,0:20])
print(adata.layers['log_counts'][0:20,0:20])

# check that you get integers when you un-normalize the data - randomly checking index 3 
unlog1p = unlog1p = np.expm1(adata.X[3, :])
print(unlog1p)
count_check = unlog1p/10000*count_sum[3].item()
print(count_check)

In [ ]:
# Calculate and plot highly variable genes 
sc.pp.highly_variable_genes(adata, min_mean=0.0125, max_mean=3, min_disp=0.5)
sc.pl.highly_variable_genes(adata)

# Store a copy of adata in the .raw field before subsetting to just variable genes (NOT RAW COUNTS)
adata.raw = adata

# Subset to just variable genes 
# Scale the data 
adata = adata[:, adata.var.highly_variable]
sc.pp.scale(adata, max_value=10)  

# Run PCA and generate PCA plots 
sc.tl.pca(adata, svd_solver='arpack')
sc.pl.pca(adata, color='n_genes')
sc.pl.pca_variance_ratio(adata, log=True)

# Calculate nearest neighbors 
sc.pp.neighbors(adata, n_neighbors=10, n_pcs=40)

# Calculate
sc.tl.umap(adata)

# Calculate and plot leiden clusters 
sc.tl.leiden(adata, resolution=.1, key_added = 'leiden_res.1')
sc.pl.umap(adata, color=['leiden_res.1'], size=1)

adata.write_h5ad(os.path.join(base_path, f'data/yocrc_{cell_type}_processed.h5ad'))

In [ ]:
# Optional jump in point 
adata = sc.read_h5ad(os.path.join(base_path, f'data/yocrc_{cell_type}_processed.h5ad'))

In [ ]:
# Plot FRID
import random
colors = ["#%06x" % random.randint(0, 0xFFFFFF) for _ in range(192)]
sc.pl.umap(adata, color='FRID', size=5, palette=colors)

In [ ]:
# Integrate with harmony (patient batch) 
import scanpy.external as sce
sce.pp.harmony_integrate(adata, key="FRID")
sc.pp.neighbors(adata, n_neighbors=10, n_pcs=40, use_rep='X_pca_harmony')
sc.tl.umap(adata)

import random
colors = ["#%06x" % random.randint(0, 0xFFFFFF) for _ in range(192)]
sc.pl.umap(adata, color='FRID', size=5, palette=colors)

In [ ]:
adata.write_h5ad(os.path.join(base_path, f'data/yocrc_{cell_type}_harmony.h5ad'))

In [ ]:
# Optional jump in point 
adata = sc.read_h5ad(os.path.join(base_path, f'data/yocrc_{cell_type}_harmony.h5ad'))

In [ ]:
# Calculate and visualize different clustering resolutions 
sc.tl.leiden(adata, resolution=.05, key_added='harmony_leiden_res.05')
sc.tl.leiden(adata, resolution=.1, key_added='harmony_leiden_res.1')
sc.tl.leiden(adata, resolution=.2, key_added='harmony_leiden_res.2')
sc.tl.leiden(adata, resolution=.3, key_added='harmony_leiden_res.3')
sc.tl.leiden(adata, resolution=.4, key_added='harmony_leiden_res.4')
sc.tl.leiden(adata, resolution=.5, key_added='harmony_leiden_res.5')
sc.tl.leiden(adata, resolution=.6, key_added='harmony_leiden_res.6')
sc.tl.leiden(adata, resolution=.8, key_added='harmony_leiden_res.8')
sc.tl.leiden(adata, resolution=.8, key_added='harmony_leiden_res.9')
sc.tl.leiden(adata, resolution=1, key_added='harmony_leiden_res1')
sc.tl.leiden(adata, resolution=1.5, key_added='harmony_leiden_res1.5')
sc.pl.umap(adata, color=['harmony_leiden_res.05', 'harmony_leiden_res.1', 'harmony_leiden_res.2', 'harmony_leiden_res.3', 
                         'harmony_leiden_res.4', 'harmony_leiden_res.5', 'harmony_leiden_res.6', 'harmony_leiden_res.8', 
                         'harmony_leiden_res.9', 'harmony_leiden_res1', 'harmony_leiden_res1.5'], size=4)

In [ ]:
# Save harmony integrated object 
adata.write_h5ad(os.path.join(base_path, f'data/yocrc_{cell_type}_harmony.h5ad'))

In [ ]:
# Run cell typist
import celltypist
from celltypist import models

#used as reference for comparing manual annotations based on sigantures
predictions = celltypist.annotate(
    adata.raw.to_adata(), 
    model = 'Cells_Intestinal_Tract.pkl', 
    majority_voting = True
)

predict = predictions.to_adata()
tmp = predict.obs['majority_voting'].astype("str")
adata.obs['majority_voting']=tmp
sc.pl.umap(
    adata, color='majority_voting', wspace=0.1, add_outline=False, size=6, 
    legend_fontsize=11, legend_fontoutline=1.5, frameon=False, palette='tab20')

In [ ]:
# Run cell typist
import celltypist
from celltypist import models

#used as reference for comparing manual annotations based on sigantures
predictions = celltypist.annotate(
    adata.raw.to_adata(), 
    model = 'Human_Colorectal_Cancer.pkl', 
    majority_voting = True
)

predict = predictions.to_adata()
tmp = predict.obs['majority_voting'].astype("str")
adata.obs['majority_voting']=tmp
sc.pl.umap(
    adata, color='majority_voting', wspace=0.1, add_outline=False, size=6, 
    legend_fontsize=11, legend_fontoutline=1.5, frameon=False, palette='tab20')

In [ ]:
# Calculate clsuter marker genes 
sc.tl.rank_genes_groups(adata, "harmony_leiden_res.2", method="wilcoxon", use_raw = True)

# save marker genes 
df_all = pd.DataFrame()
clusters = np.unique(adata.obs['harmony_leiden_res.2'])
for i in clusters:
    
    df = sc.get.rank_genes_groups_df(adata, group = i)

    #order by zscore
    df['abs_logFC'] = np.absolute(df['logfoldchanges'])
    df = df[['names', 'scores', 'logfoldchanges', 'abs_logFC', 'pvals', 'pvals_adj']]
    print(i)
    print(df)
    
    i = i.replace(" / ", "_")
    i = i.replace(" ", "_")
        
    tmp=df['names'][0:100]
    df_all = pd.concat([df_all, tmp], axis=1)
df_all.columns=clusters
df_all.to_csv(os.path.join(base_path, f"results/{cell_type}_res.2_markerGenes_top100.csv"))

In [ ]:
# Look at top marker genes of each cluster 
for i in range(len(df_all.columns)):
    print(i)
    sc.pl.umap(adata, color=df_all[str(i)][0:20], ncols=10)

In [ ]:
# Look at top marker genes of each cluster - res.3
df_all = pd.read_csv(os.path.join(base_path, f"results/{cell_type}_res.3_markerGenes_top100.csv"))
for i in range(len(df_all.columns)-2):
    print(i)
    sc.pl.umap(adata, color=df_all[str(i)][0:20], ncols=10)

In [ ]:
# optional jump in point 
adata = sc.read_h5ad(os.path.join(base_path, f'data/yocrc_{cell_type}_harmony.h5ad'))

In [ ]:
# Cycling score 
cell_cycle_genes = [x.strip() for x in open(os.path.join(base_path, 'docs/regev_lab_cell_cycle_genes.txt'))]
s_genes = cell_cycle_genes[:43]
g2m_genes = cell_cycle_genes[43:]
sc.tl.score_genes_cell_cycle(adata, s_genes=s_genes, g2m_genes=g2m_genes)
sc.pl.umap(adata, color=['phase'])
sc.pl.violin(adata, 'phase', groupby='harmony_leiden_res.3')

In [ ]:
# Load original adata 
adata = sc.read_h5ad(os.path.join(base_path, f'data/yocrc_{cell_type}_harmony.h5ad'))

In [ ]:
sc.pl.umap(adata, color=['IGHM', 'IGHD', 'CD27', 'CD38', 'CD24', 'CD80', 'CD86'], size=4)

In [ ]:
# Assign cluster annotations 
tier2annotation = {
    '0' : 'B cell',
    '1' : 'Plasma cell', 
    '2' : 'Mixed - B cell', # Epi 
    '3' : 'Mixed - B cell', # ? 
    '4' : 'Germinal center / Cycling B cell', #
    '5' : 'Mixed - B cell', # T 
    '6' : 'Cycling plasma cell', #
    '7' : 'HSP-hi - B cell', #
}
adata.obs['Annotation_Tier2'] = adata.obs['harmony_leiden_res.2'].map(tier2annotation).astype('category')

In [ ]:
sc.pl.umap(adata, color=['Annotation_Tier2'], size=4)

In [ ]:
# Save annotated adata 
adata.write_h5ad(os.path.join(base_path, f'data/yocrc_{cell_type}_annotation.h5ad'))

In [ ]:
# read object in 
adata = sc.read_h5ad(os.path.join(base_path, f'data/yocrc_{cell_type}_annotation.h5ad'))

In [ ]:
# Plot Tier2
colors = ['#000000', '#E69F00', '#0072B2', '#D55E00', '#009E73', '#56B4E9', '#CC79A7','#F0E442','#604882', '#ACAF7A']

fig, ax = plt.subplots()
sc.pl.umap(adata, color='Annotation_Tier2', size=3, ax=ax, show=True, palette=colors)
fig.savefig(os.path.join(base_path, f'results/annotations_EOCRC_DATE/Annotation_Tier2_UMAP_{cell_type}_Harmony.pdf'), dpi=600, bbox_inches='tight')
plt.close(fig)

In [ ]:
# Plot FRID
import random
colors = ["#%06x" % random.randint(0, 0xFFFFFF) for _ in range(192)]

fig, ax = plt.subplots()
sc.pl.umap(adata, color='FRID', size=3, ax=ax, show=True, palette=colors)
fig.savefig(os.path.join(base_path, f'results/annotations_EOCRC_DATE/FRID_UMAP_{cell_type}_Harmony.pdf'), dpi=600, bbox_inches='tight')
plt.close(fig)

In [ ]:
adata_orig = sc.read_h5ad(os.path.join(base_path, f'data/yocrc_{cell_type}_processed.h5ad'))

In [ ]:
# check that sizes check out 
print(sum(adata_orig.obs_names==adata.obs_names))
print(adata.shape)
print(adata_orig.shape)

In [ ]:
# transfer obsm and obs to the raw data object
adata_orig.obs = adata.obs 

In [ ]:
adata_orig.write_h5ad(os.path.join(base_path, f'data/yocrc_{cell_type}_annotation_noHarmony.h5ad'))

In [ ]:
# Plot Tier2
colors = ['#000000', '#E69F00', '#0072B2', '#D55E00', '#009E73', '#56B4E9', '#CC79A7','#F0E442','#604882', '#ACAF7A']

fig, ax = plt.subplots()
sc.pl.umap(adata_orig, color='Annotation_Tier2', size=3, ax=ax, show=True, palette=colors)
fig.savefig(os.path.join(base_path, f'results/annotations_EOCRC_DATE/Annotation_Tier2_UMAP_{cell_type}_NoHarmony.pdf'), dpi=600, bbox_inches='tight')
plt.close(fig)

In [ ]:
# Plot and save metadata on UMAP
colors = ['#000000', '#E69F00', '#56B4E9', '#009E73', '#F0E442', '#0072B2', '#D55E00', '#CC79A7']
sc.set_figure_params(figsize=(4, 4))

# Plot DECADE
fig, ax = plt.subplots()
sc.pl.umap(adata_orig, color='Decade', size=3, ax=ax, show=True, palette=colors)
fig.savefig(os.path.join(base_path, f'results/annotations_EOCRC_DATE/Decade_UMAP_{cell_type}_NoHarmony.pdf'), dpi=600, bbox_inches='tight')
plt.close(fig)

# Plot AGE COHORT
fig, ax = plt.subplots()
sc.pl.umap(adata_orig, color='Cohort', size=3, ax=ax, show=True, palette=colors[5:7])
fig.savefig(os.path.join(base_path, f'results/annotations_EOCRC_DATE/Cohort_UMAP_{cell_type}_NoHarmony.pdf'), dpi=600, bbox_inches='tight')
plt.close(fig)

# Plot FRID
import random
colors = ["#%06x" % random.randint(0, 0xFFFFFF) for _ in range(192)]

fig, ax = plt.subplots()
sc.pl.umap(adata_orig, color='FRID', size=3, ax=ax, show=True, palette=colors)
fig.savefig(os.path.join(base_path, f'results/annotations_EOCRC_DATE/FRID_UMAP_{cell_type}_NoHarmony.pdf'), dpi=600, bbox_inches='tight')
plt.close(fig)